In [ ]:
%run ./1_config
from pyspark.sql import functions as F

class BronzeLoader:
    def __init__(self):
        self.c = conf

    def load_listings(self):
        df = (spark.read.option("header", True).csv(self.c.listings_path)
              .withColumn("price", F.col("price").cast("int"))
              .withColumn("bedrooms", F.col("bedrooms").cast("int"))
              .withColumn("bathrooms", F.col("bathrooms").cast("int"))
              .withColumn("land_size_sqm", F.col("land_size_sqm").cast("int"))
              .withColumn("is_luxury", F.col("is_luxury").cast("boolean"))
              .withColumn("_ingested_at", F.current_timestamp()))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.bronze_listings))
        print(f"✅ bronze_listings: {df.count()} rows")

    def load_views(self):
        cutoff = F.date_sub(F.current_date(), self.c.lookback_days)
        df = (spark.read.option("header", True).csv(self.c.views_path)
              .withColumn("event_ts", F.to_timestamp("event_ts"))
              .withColumn("view_duration_sec", F.col("view_duration_sec").cast("int"))
              .withColumn("enquiry_flag", F.col("enquiry_flag").cast("boolean"))
              .withColumn("favorite_flag", F.col("favorite_flag").cast("boolean"))
              .withColumn("event_date", F.to_date("event_ts"))
              .withColumn("_ingested_at", F.current_timestamp())
              .filter(F.col("event_date") >= cutoff))
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(self.c.table_fqn(self.c.bronze_property_views))
        print(f"✅ bronze_property_views: {df.count()} rows appended")

    def run(self):
        self.load_listings()
        self.load_views()

BronzeLoader().run()

